# 1. Pengunduhan & Penggabungan Dataset VNetra

Notebook ini memisahkan tugas berat CPU (download COCO) dan tugas GPU (Merge & Training).


In [ ]:
!pip install -q albumentations kagglehub python-dotenv
from google.colab import drive
drive.mount('/content/drive')

import os
from dotenv import load_dotenv

# ponytail: load dari satu file .env di Google Drive, tak perlu set Colab Secrets tiap run
env_path = '/content/drive/MyDrive/YOLO/vnetra.env'
load_dotenv(env_path)
if os.path.exists(env_path):
    print(f"Loaded config from {env_path}")
else:
    print(f"Bikin file {env_path} berisi ROBOFLOW_API_KEY=... dan KAGGLE_USERNAME=... dll")

# --- KONFIGURASI EKSPERIMEN ---
EXPERIMENT_ID = 18

DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import IPython
import PIL
pil_ver = PIL.__version__
IPython.get_ipython().system(f"pip install -q ultralytics roboflow pyyaml Pillow=={pil_ver}")

import importlib, site
importlib.reload(site)
importlib.invalidate_caches()
print("Environment siap!")


Mounted at /content/drive
Mengunci versi Pillow ke 11.3.0 untuk mencegah crash C-extension...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 72.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
Environment siap!


## 1. Unduh Dataset

### Perubahan di Eksperimen 18:
- ❌ **COCO standar dihapus** — perspektif foto tidak sesuai dengan kamera VNetra
- ✅ **BDD100K ditambahkan** — dataset dashcam, perspektif paling mirip dengan kamera kepala VNetra
- ✅ **Dataset Lalu-lintas Indonesia** — konteks jalan Indonesia (angkot, ojek, pejalan kaki)
- ✅ Dataset kustom lainnya (tactile, stairs, tree, crosswalk, pole) **dipertahankan**

> **Catatan BDD100K:** Upload BDD100K (format YOLO dari Kaggle) secara manual ke Google Drive
> di path: `/content/drive/MyDrive/YOLO/bdd100k_yolo/`
> Download dari: https://www.kaggle.com/datasets/solesensei/solesensei_bdd100k


In [ ]:
import os
from roboflow import Roboflow

# ponytail: baca langsung dari environment (diisi oleh load_dotenv tadi)
roboflow_key = os.environ.get('ROBOFLOW_API_KEY')
if not roboflow_key:
    raise ValueError("Set ROBOFLOW_API_KEY di vnetra.env!")

rf = Roboflow(api_key=roboflow_key)

# =====================================================================
# 1. BDD100K — Download langsung dari Kaggle via kagglehub
# =====================================================================
import kagglehub

print("Mencari BDD100K...")
# Prioritaskan dataset lokal di Google Drive (format YOLO bounding box yang valid)
BDD100K_PATH = '/content/drive/MyDrive/YOLO/bdd100k_yolo'

if not os.path.exists(BDD100K_PATH):
    print("BDD100K tidak ada di Drive. Mencoba unduh dari Kaggle...")
    try:
        # PENTING: Gunakan dataset YOLO (bukan yang _seg / segmentation)
        # BDD100K yang valid biasanya bernama bdd100k-yolo-format, bukan solesensei (segmentasi)
        BDD100K_PATH = kagglehub.dataset_download("yaminh/bdd100k-yolo") 
        print(f"BDD100K dari Kaggle siap di: {BDD100K_PATH}")
    except Exception as e:
        print(f"Gagal mengunduh dataset alternatif dari Kaggle: {e}")
        print("Silakan masukkan BDD100K_PATH yang valid secara manual.")
else:
    print(f"Menggunakan BDD100K dari Google Drive: {BDD100K_PATH}")


# =====================================================================
# 2. Dataset Lalu-lintas Indonesia (Roboflow)
# =====================================================================
print("\nMengunduh Dataset Lalu-lintas Indonesia...")
dataset_indonesia = None
candidates = [
    ("itn-bandung", "deteksi-kendaraan-indonesia", 1),
    ("roboflow-100", "vehicles-q0x2v", 2),
]
for workspace, project, version in candidates:
    try:
        dataset_indonesia = rf.workspace(workspace).project(project).version(version).download("yolov11")
        print(f"  Dataset Indonesia OK: {workspace}/{project}")
        break
    except Exception as e:
        print(f"  Skip {workspace}/{project}: {e}")

# =====================================================================
# 3. Dataset Kustom VNetra (navigasi tunanetra) — TETAP
# =====================================================================
print("\nMengunduh dataset kustom VNetra...")
dataset_tactile   = rf.workspace("raihan-aria").project("paving-tactile-detection").version(4).download("yolov11")
dataset_pole      = rf.workspace("ghost-gsj7h").project("utility-pole-aka9k").version(3).download("yolov11")
dataset_stairs    = rf.workspace("jatin-sne2e").project("stairs-zqsvn").version(2).download("yolov11")
dataset_stairs2   = rf.workspace("sovar-sfwov").project("stair-detection-large").version(1).download("yolov11")
dataset_tree      = rf.workspace("tree-nqhzs").project("tree-hmf5d").version(1).download("yolov11")
dataset_crosswalk = rf.workspace("wqwdas").project("crosswalk-1elwe").version(1).download("yolov11")

print("\nSemua dataset siap!")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to COCO-Dataset-46 in yolov11:: 100%|██████████| 245853/245853 [01:59<00:00, 2058.33it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Paving-Tactile-Detection-4 in yolov11:: 100%|██████████| 4292/4292 [00:01<00:00, 4194.24it/s]


loading Roboflow workspace...
loading Roboflow project...


...

loading Roboflow project...



Extracting Dataset Version Zip to tree-1 in yolov11:: 100%|██████████| 6776/6776 [00:03<00:00, 2118.44it/s]

loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to crosswalk--1 in yolov11:: 100%|██████████| 6147/6147 [00:02<00:00, 2256.85it/s]


## 2. Penggabungan (Merging) Seluruh Dataset
Menyatukan seluruh dataset (11+ Roboflow + 1 COCO) ke dalam folder `vnetra_master_dataset` sambil merekayasa ID Kelas mereka agar berurutan (0-22) secara konsisten dan membatasi jumlah maksimal objek per kelas (Rebalancing).


In [ ]:

def extract_if_zip(source_path):
    import zipfile
    for root, _, files in os.walk(source_path):
        for f in files:
            if f.endswith('.zip'):
                zip_path = os.path.join(root, f)
                print(f"  [DEBUG] Menemukan file zip: {zip_path}, mengekstrak...")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(root)
                print(f"  [DEBUG] Ekstrak selesai!")
                return True
    return False
import os
import shutil
import yaml
import glob
import random

master_dir = "/content/vnetra_master_dataset"
if os.path.exists(master_dir): shutil.rmtree(master_dir)
os.makedirs(f"{master_dir}/train/images", exist_ok=True)
os.makedirs(f"{master_dir}/train/labels", exist_ok=True)
os.makedirs(f"{master_dir}/valid/images", exist_ok=True)
os.makedirs(f"{master_dir}/valid/labels", exist_ok=True)
os.makedirs(f"{master_dir}/test/images", exist_ok=True)
os.makedirs(f"{master_dir}/test/labels", exist_ok=True)

# === DEFINISI KELAS MASTER (14 kelas, sama seperti sebelumnya) ===
master_classes = [
    "person", "car", "motorcycle", "bus", "pole",
    "tactile_paving_straight", "tactile_paving_turn",
    "tactile_paving_3way", "tactile_paving_4way", "tactile_paving_stop",
    "stairs_up", "stairs_down", "crosswalk", "tree"
]
master_class_to_id = {name: idx for idx, name in enumerate(master_classes)}

global_counter = {}
global_image_counter = {}

# === LIMIT PER KELAS (disesuaikan untuk dataset baru) ===
# BDD100K + Indonesia: lebih sedikit tapi perspektif lebih tepat
VEHICLE_CLASS_LIMITS = {
    "person":     3000,   # Kurangi dari 6000 - kualitas > kuantitas
    "car":        3000,   # Kurangi dari 5000
    "truck":      1000,
    "bus":        1500,
    "motorcycle": 2000,
    "rider":      1000,   # BDD100K: rider = orang di atas motor
}

MAX_INSTANCES = 3

# =====================================================================
# FUNGSI MERGE (REVISI - filter diperbaiki)
# =====================================================================
def merge_dataset(source_path, class_mapping, max_samples=None, max_instances_per_class=None, max_samples_per_class=None, instance_counter=None):
    print(f"[DEBUG] Memulai merge dari: {source_path}")
    extract_if_zip(source_path)
    if instance_counter is None:
        instance_counter = {}

    yaml_path = os.path.join(source_path, 'data.yaml')
    if not os.path.exists(yaml_path):
        # ponytail: cari data.yaml di dalam subfolder (sering terjadi di kagglehub)
        found = False
        for root, _, files in os.walk(source_path):
            yaml_files = [f for f in files if f.endswith('.yaml') or f.endswith('.yml')]
            if yaml_files:
                yaml_path = os.path.join(root, yaml_files[0])
                # JANGAN UBAH source_path! Biarkan Ultimate Path Resolver mencari di seluruh tree
                found = True
                break
        
        # BDD100K KAGGLEHUB FALLBACK: Jika tidak ada yaml sama sekali, buatkan satu!
        if not found and 'bdd100k' in source_path.lower():
            yaml_path = os.path.join(source_path, 'data.yaml')
            # BDD100K YOLO Format (solesensei)
            fake_yaml = "names: ['person', 'rider', 'car', 'bus', 'truck', 'bike', 'motor', 'traffic light', 'traffic sign', 'train']\n"
            with open(yaml_path, 'w') as yf: yf.write(fake_yaml)
            found = True
        if not found:
            print(f"  SKIP: data.yaml tidak ditemukan di {source_path} atau subfoldernya")
            return

    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)
        original_classes = yaml_data.get('names', [])
        if isinstance(original_classes, dict):
            original_classes = [original_classes[k] for k in sorted(original_classes.keys())]
        
        # ponytail: fuzzy matching untuk mengatasi spasi, huruf besar, dll
        clean_classes = []
        for cls in original_classes:
            clean = str(cls).lower().replace(' ', '_').replace('-', '_')
            clean_classes.append(clean)
        
        # Mapping yang masuk juga di-clean
        clean_mapping = {}
        for k, v in class_mapping.items():
            clean_mapping[k.lower().replace(' ', '_').replace('-', '_')] = v
        class_mapping = clean_mapping

    copied_count = 0
    for split in ['train', 'valid', 'test']:
        alt_split = 'val' if split == 'valid' else split
        
        img_dir = None
        lbl_dir_base = None
        
        # Ultimate Path Resolver: cari folder dengan nama 'train'/'val' atau 'images'/'labels' di seluruh tree
        for root, dirs, files in os.walk(source_path):
            parent = os.path.basename(os.path.dirname(root))
            basename = os.path.basename(root)
            
            target_splits = [split, alt_split]
            
            # Kasus 1: images/train, images/100k/train, dll
            if basename in target_splits and 'images' in root:
                img_dir = root
            # Kasus 2: labels/train, labels/100k/train, dll
            if basename in target_splits and 'labels' in root:
                lbl_dir_base = root
                
            # Kasus 3: train/images
            if parent in target_splits and basename == 'images':
                img_dir = root
            # Kasus 4: train/labels
            if parent in target_splits and basename == 'labels':
                lbl_dir_base = root
                
        if not img_dir or not lbl_dir_base or not os.path.exists(img_dir):
            print(f"  [DEBUG] SKIP split '{split}' -> tidak menemukan kombinasi folder images & labels di dalam {source_path}")
            continue
        else:
            print(f"  [DEBUG] FOUND split '{split}' -> img_dir: {img_dir}, lbl_dir: {lbl_dir_base}")

        all_images = glob.glob(f"{img_dir}/*")

        if max_samples_per_class is not None:
            scored_images = []
            for img_path in all_images:
                lbl_name = os.path.basename(img_path).rsplit('.', 1)[0] + '.txt'
                lbl_path = f"{lbl_dir_base}/{lbl_name}"
                score = random.random()
                if os.path.exists(lbl_path):
                    lines = open(lbl_path).readlines()
                    classes_in_img = set()
                    for p in lines:
                        parts = p.split()
                        if parts and int(parts[0]) < len(original_classes):
                            classes_in_img.add(str(original_classes[int(parts[0])]).lower())
                    # Prioritaskan kelas minoritas
                    if classes_in_img.intersection({'motorcycle', 'motor', 'rider', 'bus'}):
                        score += 100
                scored_images.append((score, img_path))
            scored_images.sort(key=lambda x: x[0], reverse=True)
            all_images = [p for s, p in scored_images]
        else:
            random.shuffle(all_images)

        for img_path in all_images:
            if max_samples is not None and copied_count >= max_samples:
                return

            file_name = os.path.basename(img_path)
            lbl_name = file_name.rsplit('.', 1)[0] + '.txt'
            lbl_path = f"{lbl_dir_base}/{lbl_name}"
            if not os.path.exists(lbl_path):
                continue

            lines = open(lbl_path).readlines()

            parsed_objs = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                orig_id = int(parts[0])
                if orig_id >= len(original_classes):
                    continue
                c_name = str(original_classes[orig_id]).lower()
                mapped_master = next((v for k, v in class_mapping.items() if k.lower() == c_name), None)
                if mapped_master:
                    parsed_objs.append({
                        'orig_class': c_name, 'master_class': mapped_master,
                        'box': [float(x) for x in parts[1:5]], 'line_parts': parts, 'dropped': False
                    })

            # =========================================================
            # FILTER UKURAN MINIMUM - REVISI (jauh lebih longgar)
            # Sebelumnya: person=139px → sering membuang orang jauh/kecil
            # Sekarang: person=25px → deteksi lebih awal = lebih aman untuk tunanetra
            # =========================================================
            MIN_PIXEL_SIZE = {
                'person':     25,   # ← dari 139: deteksi orang dari jauh
                'motorcycle': 15,   # ← dari 35
                'car':        25,   # ← dari 60
                'bus':        30,   # ← dari 80
            }
            for obj in parsed_objs:
                if obj['master_class'] in MIN_PIXEL_SIZE:
                    xc, yc, w, h = obj['box']
                    if max(w * 640.0, h * 640.0) < MIN_PIXEL_SIZE[obj['master_class']]:
                        obj['dropped'] = True

            # =========================================================
            # SPATIAL PASSENGER FILTER - DINONAKTIFKAN
            # Alasan: Orang yang berada dekat kendaraan adalah informasi
            # navigasi KRITIS untuk pengguna tunanetra. Filter ini
            # secara tidak sengaja menghapus skenario paling berbahaya.
            # =========================================================
            # (blok filter dihapus sepenuhnya)

            # --- PER-CLASS LIMIT ENFORCEMENT ---
            if max_samples_per_class is not None:
                for obj in parsed_objs:
                    if (not obj['dropped'] and
                        instance_counter.get(obj['orig_class'], 0) >=
                        max_samples_per_class.get(obj['orig_class'], float('inf'))):
                        obj['dropped'] = True

            valid_objs = [o for o in parsed_objs if not o['dropped']]
            if not valid_objs:
                continue

            # Salin gambar dan label
            out_split = split if split != 'valid' else 'valid'
            dst_img_dir = f"{master_dir}/{out_split}/images"
            dst_lbl_dir = f"{master_dir}/{out_split}/labels"
            os.makedirs(dst_img_dir, exist_ok=True)
            os.makedirs(dst_lbl_dir, exist_ok=True)

            # Cegah nama file duplikat
            base_name = os.path.splitext(file_name)[0]
            src_prefix = os.path.basename(source_path).replace(' ', '_')[:15]
            unique_name = f"{src_prefix}_{base_name}"
            dst_img = os.path.join(dst_img_dir, unique_name + os.path.splitext(file_name)[1])
            dst_lbl = os.path.join(dst_lbl_dir, unique_name + '.txt')

            if os.path.exists(dst_img):
                continue

            shutil.copy2(img_path, dst_img)

            with open(dst_lbl, 'w') as f:
                for obj in valid_objs:
                    new_id = master_class_to_id[obj['master_class']]
                    xc, yc, w, h = obj['box']
                    f.write(f"{new_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
                    mc = obj['orig_class']
                    instance_counter[mc] = instance_counter.get(mc, 0) + 1

            copied_count += 1

    return copied_count


# =====================================================================
# PROSES MERGE SEMUA DATASET
# =====================================================================

# 1. BDD100K — Pengganti utama COCO untuk person/car/motorcycle/bus
# BDD100K_PATH sudah didownload via kagglehub di Cell 3
BDD100K_PATH = BDD100K_PATH if 'BDD100K_PATH' in dir() else '/tmp/bdd100k'
if os.path.exists(BDD100K_PATH):
    print("Memproses BDD100K (Person, Car, Motorcycle, Bus, Rider)...")
    bdd100k_mapping = {
        'person':     'person',
        'rider':      'person',      # rider = orang di motor → tetap 'person'
        'car':        'car',
        'motor':      'motorcycle',  # BDD100K pakai 'motor'
        'motorcycle': 'motorcycle',
        'bus':        'bus',
        'truck':      'car',         # opsional: truck → car
    }
    merge_dataset(
        BDD100K_PATH,
        bdd100k_mapping,
        max_samples_per_class=VEHICLE_CLASS_LIMITS,
        instance_counter=global_counter
    )
else:
    print("SKIP BDD100K - folder tidak ditemukan. Pastikan sudah di-upload ke Drive!")

# 2. Dataset Lalu-lintas Indonesia (dari Roboflow)
indonesia_datasets = []
if 'dataset_indonesia' in dir() and dataset_indonesia is not None:
    indonesia_datasets.append(dataset_indonesia.location)

for path in indonesia_datasets:
    if path and os.path.exists(path):
        print(f"Memproses Dataset Indonesia: {path}...")
        indonesia_mapping = {
            'person': 'person',
            'pejalan kaki': 'person',
            'pedestrian': 'person',
            'car': 'car',
            'mobil': 'car',
            'sedan': 'car',
            'suv': 'car',
            'motorcycle': 'motorcycle',
            'motor': 'motorcycle',
            'sepeda motor': 'motorcycle',
            'bus': 'bus',
            'angkot': 'bus',       # Angkutan kota → bus
            'mikrolet': 'bus',
        }
        merge_dataset(
            path, indonesia_mapping,
            max_samples=3000,
            max_samples_per_class=VEHICLE_CLASS_LIMITS,
            instance_counter=global_counter
        )

# 3. Dataset Kustom VNetra (TETAP SAMA - kelas navigasi)
print("\nMemproses Dataset Tactile Paving...")
tactile_mapping = {
    'straight': 'tactile_paving_straight', 'go': 'tactile_paving_straight', '1': 'tactile_paving_straight', '0': 'tactile_paving_straight',
    'turn': 'tactile_paving_turn', '2': 'tactile_paving_turn',
    '3way': 'tactile_paving_3way', '3': 'tactile_paving_3way',
    '4way': 'tactile_paving_4way', '4': 'tactile_paving_4way',
    'stop': 'tactile_paving_stop',
}

# =====================================================================
# FUNGSI MERGE DYNAMIC (diambil dari FIX)
# =====================================================================
def merge_dynamic(source_path, target_class, instance_counter=None):
    if instance_counter is None: instance_counter = {}
    yaml_path = os.path.join(source_path, 'data.yaml')
    
    # Recursive search
    if not os.path.exists(yaml_path):
        for root, _, files in os.walk(source_path):
            yaml_files = [f for f in files if f.endswith('.yaml') or f.endswith('.yml')]
            if yaml_files:
                yaml_path = os.path.join(root, yaml_files[0])
                # JANGAN UBAH source_path!
                break
                
    if not os.path.exists(yaml_path): return
    
    with open(yaml_path, 'r') as f:
        classes = yaml.safe_load(f).get('names', [])
        
    if isinstance(classes, dict):
        classes = [classes[k] for k in sorted(classes.keys())]
        
    cmap = {str(c): target_class for c in classes if str(c).lower() != "null"}
    merge_dataset(source_path, cmap, instance_counter=instance_counter)

if 'dataset_tactile' in dir():
    merge_dataset(dataset_tactile.location, tactile_mapping, instance_counter=global_counter)

print("Memproses Dataset Pole...")
if 'dataset_pole' in dir():
    merge_dynamic(dataset_pole.location, 'pole', instance_counter=global_counter)

print("Memproses Dataset Stairs...")
stairs_mapping = {
    'stairs_up': 'stairs_up', 'up_stairs': 'stairs_up', 'stair_up': 'stairs_up',
    'stairs_down': 'stairs_down', 'down_stairs': 'stairs_down', 'stair_down': 'stairs_down',
    'stairs': 'stairs_up',  # fallback: tangga tanpa arah → stairs_up
}
if 'dataset_stairs' in dir():
    merge_dynamic(dataset_stairs.location, 'stairs_up', instance_counter=global_counter) # Asumsikan semua tangga di dataset ini adalah up (fallback)
if 'dataset_stairs2' in dir():
    merge_dynamic(dataset_stairs2.location, 'stairs_down', instance_counter=global_counter) # Asumsikan semua tangga 2 adalah down

print("Memproses Dataset Tree...")
if 'dataset_tree' in dir():
    merge_dynamic(dataset_tree.location, 'tree', instance_counter=global_counter)

print("Memproses Dataset Crosswalk...")
if 'dataset_crosswalk' in dir():
    merge_dynamic(dataset_crosswalk.location, 'crosswalk', instance_counter=global_counter)

# =====================================================================
# BUAT data.yaml
# =====================================================================
data_yaml = {
    'path': master_dir,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(master_classes),
    'names': master_classes
}
with open(f'{master_dir}/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

# =====================================================================
# LAPORAN STATISTIK
# =====================================================================
print("\n" + "="*55)
print("Merge selesai! Statistik Instance per kelas:")
import glob as _glob
def count_instances(label_dir, cls_id):
    count = 0
    for f in _glob.glob(f"{label_dir}/**/*.txt", recursive=True):
        for line in open(f).readlines():
            parts = line.strip().split()
            if parts and int(parts[0]) == cls_id:
                count += 1
    return count

for idx, cls_name in enumerate(master_classes):
    n = count_instances(f"{master_dir}/train/labels", idx)
    print(f"  {cls_name:<30}: {n:>6,} instance (train)")
print("="*55)


Memproses Dataset COCO dari Roboflow (Person, Car, Bus, Truck, Motorcycle, Bicycle)...
Melakukan pre-scan pada 87582 gambar untuk mengutamakan kelas minoritas...
Melakukan pre-scan pada 23544 gambar untuk mengutamakan kelas minoritas...
Melakukan pre-scan pada 11798 gambar untuk mengutamakan kelas minoritas...
Memproses Tactile Paving Dataset...
Memproses Pole Dataset...
Memproses Stairs Dataset...
Memproses Tree Dataset...
Memproses Crosswalk Dataset...

✅ Merge selesai! Statistik Gambar & Instance per kelas:
  person              :  3,662 gambar  |   6,000 instance
  car                 :  3,657 gambar  |   5,000 instance
  motorcycle          :  1,565 gambar  |   2,000 instance
  bus                 :  1,189 gambar  |   1,500 instance
  pole                :    200 gambar  |     321 instance
  tactile_paving_straight:    859 gambar  |   1,810 instance
  tactile_paving_turn :    556 gambar  |     556 instance
  tactile_paving_3way :    618 gambar  |     618 instance
  tactile_paving_

### 2.1 Jaring Pengaman Rebalancing Data
Mendistribusikan secara adil jumlah gambar (15%) ke dalam keranjang Validation dan Test set, sambil mengembalikan sisa gambar berlebih kembali ke Train set.

In [ ]:
import os
import random
import shutil

print("=== MEMASTIKAN DISTRIBUSI HYBRID VALIDATION & TEST SET (REBALANCING) ===")
train_img_dir = f'{master_dir}/train/images'
train_lbl_dir = f'{master_dir}/train/labels'
valid_img_dir = f'{master_dir}/valid/images'
valid_lbl_dir = f'{master_dir}/valid/labels'
test_img_dir  = f'{master_dir}/test/images'
test_lbl_dir  = f'{master_dir}/test/labels'

for dir_path in [valid_img_dir, valid_lbl_dir, test_img_dir, test_lbl_dir]:
    os.makedirs(dir_path, exist_ok=True)

master_classes = [
    "person", "car", "motorcycle", "bus", "pole",
    "tactile_paving_straight", "tactile_paving_turn",
    "tactile_paving_3way", "tactile_paving_4way", "tactile_paving_stop",
    "stairs_up", "stairs_down", "crosswalk", "tree"
]

def get_class_counts(lbl_dir):
    counts = {i: 0 for i in range(len(master_classes))}
    if not os.path.exists(lbl_dir): return counts
    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(lbl_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if cls_id in counts: counts[cls_id] += 1
    return counts

train_counts = get_class_counts(train_lbl_dir)
valid_counts = get_class_counts(valid_lbl_dir)
test_counts  = get_class_counts(test_lbl_dir)

total_counts = {}
target_valid_test = {}

for cls_id in range(len(master_classes)):
    total = train_counts[cls_id] + valid_counts[cls_id] + test_counts[cls_id]
    total_counts[cls_id] = total
    if total > 0:
        # Ambil 15% dari total, dengan batas maksimum 500 dan minimum 1
        target = max(1, int(0.15 * total))
        target_valid_test[cls_id] = min(target, 500)
    else:
        target_valid_test[cls_id] = 0

def balance_split(target_split_name, target_img_dir, target_lbl_dir, current_counts):
    classes_to_boost = [c for c in range(len(master_classes)) if current_counts[c] < target_valid_test[c]]

    if not classes_to_boost:
        print(f"Semua kelas sudah mencapai target hybrid di {target_split_name} Set! Aman.")
        return current_counts

    print(f"Ada kelas yang kurang data di {target_split_name} Set: {classes_to_boost}")
    print(f"Meminjam gambar secara acak dari folder Train untuk {target_split_name}...")

    train_labels = [f for f in os.listdir(train_lbl_dir) if f.endswith('.txt')]
    random.shuffle(train_labels)

    moved_images = 0
    for lbl_file in train_labels:
        if not classes_to_boost: break

        src_lbl = os.path.join(train_lbl_dir, lbl_file)
        lines = open(src_lbl).readlines()
        classes_in_file = {int(line.split()[0]) for line in lines if line.strip()}
        contains_needed_class = any(c in classes_to_boost for c in classes_in_file)

        if contains_needed_class:
            dst_lbl = os.path.join(target_lbl_dir, lbl_file)
            img_file_base = os.path.splitext(lbl_file)[0]

            src_img, dst_img = None, None
            for ext in ['.jpg', '.jpeg', '.png']:
                temp_src = os.path.join(train_img_dir, img_file_base + ext)
                if os.path.exists(temp_src):
                    src_img = temp_src
                    dst_img = os.path.join(target_img_dir, img_file_base + ext)
                    break

            if src_img and os.path.exists(src_img):
                shutil.move(src_img, dst_img)
                shutil.move(src_lbl, dst_lbl)
                moved_images += 1

                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        c_id = int(parts[0])
                        if c_id in current_counts:
                            current_counts[c_id] += 1

                classes_to_boost = [c for c in range(len(master_classes)) if current_counts[c] < target_valid_test[c]]

    print(f"Berhasil memindahkan {moved_images} gambar dari Train ke {target_split_name}!")
    return current_counts

print("\n--- 1. HYBRID REBALANCING VALIDATION SET ---")
valid_counts = balance_split("Validation", valid_img_dir, valid_lbl_dir, valid_counts)

print("\n--- 2. HYBRID REBALANCING TEST SET ---")
test_counts = balance_split("Test", test_img_dir, test_lbl_dir, test_counts)

print("\\n=== MENGEMBALIKAN KELEBIHAN GAMBAR KE FOLDER TRAIN (STRICT CAPPING) ===")
train_counts = get_class_counts(train_lbl_dir)

def return_excess_to_train(source_name, source_img_dir, source_lbl_dir, current_counts):
    # DIBUANG: and train_counts[c] < current_counts[c]
    classes_to_reduce = [c for c in range(len(master_classes)) if current_counts[c] > target_valid_test[c]]

    if not classes_to_reduce:
        return current_counts

    print(f"Mengembalikan kelebihan data dari {source_name} ke Train untuk kelas: {classes_to_reduce}")

    labels_list = [f for f in os.listdir(source_lbl_dir) if f.endswith('.txt')]
    random.shuffle(labels_list)
    moved_back = 0

    for lbl_file in labels_list:
        if not classes_to_reduce: break

        src_lbl = os.path.join(source_lbl_dir, lbl_file)
        lines = open(src_lbl).readlines()
        classes_in_file = {int(line.split()[0]) for line in lines if line.strip()}
        contains_excess_class = any(c in classes_to_reduce for c in classes_in_file)

        if contains_excess_class:
            dst_lbl = os.path.join(train_lbl_dir, lbl_file)
            img_file_base = os.path.splitext(lbl_file)[0]

            src_img, dst_img = None, None
            for ext in ['.jpg', '.jpeg', '.png']:
                temp_src = os.path.join(source_img_dir, img_file_base + ext)
                if os.path.exists(temp_src):
                    src_img = temp_src
                    dst_img = os.path.join(train_img_dir, img_file_base + ext)
                    break

            if src_img and os.path.exists(src_img):
                can_move = True
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        c_id = int(parts[0])
                        if current_counts[c_id] <= target_valid_test[c_id]:
                            can_move = False
                            break

                if can_move:
                    shutil.move(src_img, dst_img)
                    shutil.move(src_lbl, dst_lbl)
                    moved_back += 1

                    for line in lines:
                        parts = line.strip().split()
                        if parts:
                            c_id = int(parts[0])
                            current_counts[c_id] -= 1
                            train_counts[c_id] += 1

                    classes_to_reduce = [c for c in range(len(master_classes)) if current_counts[c] > target_valid_test[c]]

    print(f"Berhasil mengembalikan {moved_back} gambar dari {source_name} ke Train!")
    return current_counts

valid_counts = return_excess_to_train("Validation", valid_img_dir, valid_lbl_dir, valid_counts)
test_counts = return_excess_to_train("Test", test_img_dir, test_lbl_dir, test_counts)
print("\nDistribusi Hybrid Selesai! Model akan aman dari Catastrophic Forgetting untuk kelas kecil.")



=== MEMASTIKAN DISTRIBUSI HYBRID VALIDATION & TEST SET (REBALANCING) ===

--- 1. HYBRID REBALANCING VALIDATION SET ---
Ada kelas yang kurang data di Validation Set: [0, 1, 3, 5, 7, 10, 11, 12, 13]
Meminjam gambar secara acak dari folder Train untuk Validation...
Berhasil memindahkan 1190 gambar dari Train ke Validation!

--- 2. HYBRID REBALANCING TEST SET ---
Ada kelas yang kurang data di Test Set: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Meminjam gambar secara acak dari folder Train untuk Test...
Berhasil memindahkan 1640 gambar dari Train ke Test!
\n=== MENGEMBALIKAN KELEBIHAN GAMBAR KE FOLDER TRAIN (STRICT CAPPING) ===
Mengembalikan kelebihan data dari Validation ke Train untuk kelas: [1, 2, 3, 4, 5, 6, 8, 9, 13]
Berhasil mengembalikan 260 gambar dari Validation ke Train!
Mengembalikan kelebihan data dari Test ke Train untuk kelas: [0, 1, 3, 4, 5, 6, 7, 9, 12]
Berhasil mengembalikan 159 gambar dari Test ke Train!

Distribusi Hybrid Selesai! Model akan aman dari Catastrophic Fo

### 2.2 Laporan Akhir Proporsi Dataset
Mencetak tabel distribusi dari keseluruhan dataset untuk verifikasi manual.

In [ ]:
import pandas as pd
import os

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images(f'{master_dir}/train/images')
valid_count = count_images(f'{master_dir}/valid/images')
test_count  = count_images(f'{master_dir}/test/images')
total_images = train_count + valid_count + test_count

print("=== Statistik Keseluruhan ===")
print(f"Total Lembar Gambar (All) : {total_images} gambar")
print(f"Total Gambar Training     : {train_count} gambar")
print(f"Total Gambar Validasi     : {valid_count} gambar")
print(f"Total Gambar Testing      : {test_count} gambar")
print("=============================")
print("")

def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts: counts[int(parts[0])] += 1
    return counts

train_cls = count_instances_per_class(f'{master_dir}/train/labels', len(master_classes))
valid_cls = count_instances_per_class(f'{master_dir}/valid/labels', len(master_classes))
test_cls  = count_instances_per_class(f'{master_dir}/test/labels', len(master_classes))

data_report = []
total_train = 0
total_valid = 0
total_test = 0
global_total = 0

for i, cls_name in enumerate(master_classes):
    t_train = train_cls[i]
    t_valid = valid_cls[i]
    t_test = test_cls[i]
    t_total = t_train + t_valid + t_test

    total_train += t_train
    total_valid += t_valid
    total_test += t_test
    global_total += t_total

    data_report.append({
        'ID': i,
        'Kelas': cls_name,
        'Train (Inst)': t_train,
        'Valid (Inst)': t_valid,
        'Test (Inst)': t_test,
        'Total Instance': t_total
    })

data_report.append({
    'ID': '-',
    'Kelas': 'TOTAL KESELURUHAN',
    'Train (Inst)': total_train,
    'Valid (Inst)': total_valid,
    'Test (Inst)': total_test,
    'Total Instance': global_total
})

df_report = pd.DataFrame(data_report)
display(df_report)


=== Statistik Keseluruhan ===
Total Lembar Gambar (All) : 13415 gambar
Total Gambar Training     : 9941 gambar
Total Gambar Validasi     : 1809 gambar
Total Gambar Testing      : 1665 gambar



,ID,Kelas,Train (Inst),Valid (Inst),Test (Inst),Total Instance
0,0,person,4959,500,541,6000
1,1,car,4000,500,500,5000
2,2,motorcycle,1400,300,300,2000
3,3,bus,1050,225,225,1500
4,4,pole,226,48,47,321
5,5,tactile_paving_straight,1269,270,271,1810
6,6,tactile_paving_turn,390,83,83,556
7,7,tactile_paving_3way,434,92,92,618
8,8,tactile_paving_4way,258,55,55,368
9,9,tactile_paving_stop,264,56,56,376


### 2.3 Backup Dataset ke Google Drive (Wajib untuk Resume)
Dataset yang sudah digabung akan di-ZIP dan dikirim langsung ke `INPUT_DIR` di Google Drive Anda agar notebook *Resume* dapat mengambilnya kembali jika sesi Colab ini terputus.

In [ ]:
import shutil
import os

print("📦 Membuat arsip ZIP untuk seluruh dataset master...")
dataset_dir = master_dir
zip_path = f'{INPUT_DIR}/vnetra_master_dataset'

try:
    os.makedirs(INPUT_DIR, exist_ok=True)
    shutil.make_archive(zip_path, 'zip', dataset_dir)
    print(f"✅ Selesai! Dataset master telah diamankan secara permanen ke: {zip_path}.zip")
    print("Di masa depan, Anda bisa menggunakan notebook Resume untuk memanggil dataset ini!")
except Exception as e:
    print(f"❌ Gagal melakukan backup: {e}")
    print("Pastikan Anda telah mengizinkan Google Colab untuk mengakses Google Drive Anda di cell paling atas.")


📦 Membuat arsip ZIP untuk seluruh dataset master...
✅ Selesai! Dataset master telah diamankan secara permanen ke: /content/drive/MyDrive/YOLO/eksperimen_17/input/vnetra_master_dataset.zip
Di masa depan, Anda bisa menggunakan notebook Resume untuk memanggil dataset ini!
